# Modeling Disease, Epidemiology, and Biological Spread Workflow

This notebook scaffold mirrors the repository workflow: SIR and SEIR models, Rt proxy, branching-process spread, reporting-delay adjustment, validation metrics, and provenance documentation.

In [ ]:
from pathlib import Path
import pandas as pd

article_dir = Path.cwd().parent
scenarios = pd.read_csv(article_dir / 'data' / 'model_scenarios.csv')
scenarios.head()

In [ ]:
def simulate_sir(population, initial_infected, beta, gamma, dt, steps):
    susceptible = population - initial_infected
    infected = initial_infected
    recovered = 0.0
    rows = []
    for step in range(steps + 1):
        rows.append({'step': step, 'time': step * dt, 'susceptible': susceptible, 'infected': infected, 'recovered': recovered})
        new_infections = beta * susceptible * infected / population
        new_recoveries = gamma * infected
        susceptible = max(susceptible - dt * new_infections, 0.0)
        infected = max(infected + dt * (new_infections - new_recoveries), 0.0)
        recovered = min(recovered + dt * new_recoveries, population)
    return pd.DataFrame(rows)

baseline = scenarios[scenarios['scenario'] == 'baseline'].iloc[0]
sir = simulate_sir(baseline['population'], baseline['initial_infected'], baseline['beta'], baseline['gamma'], baseline['dt'], int(baseline['steps']))
sir.tail().round(5)

In [ ]:
incidence = pd.read_csv(article_dir / 'data' / 'incidence.csv')
incidence['nowcast_cases'] = incidence['reported_cases'] / incidence['estimated_reporting_completeness']
incidence.round(4)